# Cell QC filtering for gastrula-to-pup MuData

This notebook loads the gene-filtered MuData from NB1b and applies cell-level QC filters.
The filtered output is saved as a new file (`_filtered_qc.h5mu`) for use in training (NB2).

**Input**: `gastrula_to_pup_filtered.h5mu` (9.3M cells, 3 modalities, ~317GB)
**Output**: `gastrula_to_pup_filtered_qc.h5mu` (cell-filtered)

Run on a high-memory node (800GB+).

In [ ]:
import gc
import os

import mudata as mu
import psutil

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"

In [ ]:
h5mu_path = os.path.join(DATA_DIR, "gastrula_to_pup_filtered.h5mu")
print(f"Loading {h5mu_path}...")
mdata = mu.read_h5mu(h5mu_path)
print(f"RSS after load: {psutil.Process().memory_info().rss / 1e9:.1f} GB")
print(f"\nLoaded MuData: {mdata.n_obs} cells")
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}, dtype={mdata[mod_name].X.dtype}")
print(f"\nObs columns: {list(mdata['rna'].obs.columns)}")

## QC thresholds

sci-RNA-seq3 has lower per-cell sequencing depth than 10x Chromium:
- Median UMI ~2500, median genes ~1500
- Very low mitochondrial fraction (max 0.08 in this dataset)

Thresholds chosen based on QC distributions from NB1:

| Filter | Value | Rationale |
|--------|-------|-----------|
| `n_genes_by_counts` | > 800 | Removes ~20% lowest-complexity cells |
| `total_counts` | 1,000 – 50,000 | Lower bound removes very low-quality cells; upper is safety net (max=37,756) |
| `mt_frac` | < 0.10 | Safety net (max=0.08 in data) |

**No doublet filter**: Scrublet scores only available for 69% of cells (7.9M/11.4M).

In [ ]:
n_before = mdata["rna"].n_obs

obs = mdata["rna"].obs
cell_mask = (
    (obs["n_genes_by_counts"] > 800)
    & (obs["total_counts"] > 1000)
    & (obs["total_counts"] < 50000)
    & (obs["mt_frac"] < 0.10)
)

mdata = mu.MuData({mod_name: mdata[mod_name][cell_mask.values].copy() for mod_name in mdata.mod})
gc.collect()

n_after = mdata["rna"].n_obs
print(
    f"Filtered {n_before:,} -> {n_after:,} cells ({n_before - n_after:,} removed, {(n_before - n_after) / n_before * 100:.1f}%)"
)
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}")
print(f"RSS after filtering: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

In [ ]:
h5mu_filt = os.path.join(DATA_DIR, "gastrula_to_pup_filtered_qc.h5mu")
mdata.write_h5mu(h5mu_filt)
print(f"Saved: {h5mu_filt}")
print(f"File size: {os.path.getsize(h5mu_filt) / 1e9:.1f} GB")
print("\nFinal shapes:")
for mod_name in mdata.mod:
    print(f"  {mod_name}: {mdata[mod_name].shape}")